# 09. Do analysts judge the pass frame, or the whole possession?

11% of 'pressured' (P) passes have the nearest defender ≥ 4 m away at the pass frame ("far P").
44% of them were already pressured at reception (vs 1.6% of far N), and defender speed explains little.
Hypothesis: the pressure label reflects **the possession leading up to the pass**, not only the pass frame.

For every open-play pass: nearest-defender distance over the possession window
(from the game event's start frame = reception / possession start, to the pass frame; raw broadcast
coordinates; capped at 15 s). Only games whose tracking file matches the event-file snapshot are used: at the start (`d_start`), at the pass (`d_end`), the minimum (`d_min`), and
when the minimum occurred (`t_min_before`). Code: `football.tracking.possession_distances`.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from functools import partial
from concurrent.futures import ProcessPoolExecutor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, cohen_kappa_score

from football.common import KEY, cv_predict, boot_ci, boot_diff
from football.tracking import possession_distances
from football.pressure import PFF_R_PRESSURED

OUT = os.path.join("..", "outputs")
FIG = os.path.join(OUT, "figs")
E = pd.read_csv(os.path.join(OUT, "pressure_events.csv.gz"))
print(f"open-play passes: {len(E):,}")

open-play passes: 61,012


## 1. Possession-window distances (cached, ~6 min)

In [2]:
CACHE = os.path.join(OUT, "possession_distances.csv.gz")
if not os.path.exists(CACHE):
    jobs = {g: set(d.pe_id.astype(int)) for g, d in E.groupby("game_id")}
    with ProcessPoolExecutor(6) as ex:
        parts = list(ex.map(possession_distances, jobs, jobs.values()))
    pd.DataFrame([r for p in parts for r in p]).to_csv(CACHE, index=False)
W = pd.read_csv(CACHE)
X = E.merge(W, on=KEY)
X["P"] = (X.label == "P").astype(int)
X["fail"] = (~X.complete.astype(bool)).astype(int)
X["dur"] = X.n_frames / 29.97
# In 10 of 64 games the tracking file does not match the event-file snapshot (different sync/version):
# keep games where the tracking distance at the pass frame equals the event-file distance (<5% passes off by > 0.5 m)
bad = (X.d_end - X.d1).abs() > 0.5
share_bad = bad.groupby(X.game_id).mean()
good_games = share_bad[share_bad < 0.05].index
print(f"games with consistent event/tracking coordinates: {len(good_games)} of {share_bad.size}")
X = X[X.game_id.isin(good_games)].copy()
pd.Series(good_games, name="game_id").to_csv(os.path.join(OUT, "consistent_games.csv"), index=False)
print(f"passes with window: {len(X):,};  |d_end − d1| median {(X.d_end - X.d1).abs().median():.3f} m;  "
      f"possession duration median {X.dur.median():.2f} s (one-touch passes: {(X.n_frames <= 1).mean():.1%})")
X.groupby("label")[["d_start", "d_min", "d_end", "dur"]].median().round(2)

games with consistent event/tracking coordinates: 54 of 64
passes with window: 51,218;  |d_end − d1| median 0.000 m;  possession duration median 1.17 s (one-touch passes: 28.5%)


,d_start,d_min,d_end,dur
label,,,,
A,5.48,3.45,3.51,0.97
L,6.60,2.92,3.04,1.57
N,8.85,6.14,6.46,1.30
P,3.56,1.95,2.11,0.87


## 2. Which moment reproduces the labels best? (P vs N)

In [3]:
PN = X[X.label.isin(["P", "N"])].copy()
y, g = PN.P, PN.game_id
rows = []
for name, s in [("pass frame (d_end)", -PN.d_end), ("possession start (d_start)", -PN.d_start),
                ("minimum over possession (d_min)", -PN.d_min)]:
    lo, hi = boot_ci(y, s, roc_auc_score, n=300, groups=g)
    thr = np.quantile(s, 1 - y.mean())
    rows.append(dict(moment=name, roc_auc=roc_auc_score(y, s), ci95=f"[{lo:.3f}, {hi:.3f}]",
                     kappa_matched=cohen_kappa_score(y, s > thr)))
d, ci, _ = boot_diff(y, -PN.d_min, -PN.d_end, roc_auc_score, n=300, groups=g)
moments = pd.DataFrame(rows).round(3)
print(f"AUC(d_min) − AUC(d_end) = {d:+.4f} [{ci[0]:+.4f}, {ci[1]:+.4f}]")
moments

AUC(d_min) − AUC(d_end) = -0.0157 [-0.0179, -0.0132]


,moment,roc_auc,ci95,kappa_matched
0,pass frame (d_end),0.886,"[0.873, 0.899]",0.656
1,possession start (d_start),0.799,"[0.788, 0.811]",0.434
2,minimum over possession (d_min),0.870,"[0.858, 0.884]",0.629


In [4]:
# logistic models: pass-frame distance, + possession minimum, + possession start (splines), game-grouped CV
def mk(cols):
    ct = ColumnTransformer([(c, SplineTransformer(n_knots=5), [c]) for c in cols])
    return lambda s: make_pipeline(ct, StandardScaler(), LogisticRegression(max_iter=3000))


preds = {}
for name, cols in [("d_end", ["d_end"]), ("d_end + d_min", ["d_end", "d_min"]), ("d_end + d_min + d_start", ["d_end", "d_min", "d_start"])]:
    preds[name] = cv_predict(mk(cols), PN[cols], y, g, seeds=(0,))
    print(f"{name:26s} CV ROC-AUC = {roc_auc_score(y, preds[name]):.4f}")
d, ci, _ = boot_diff(y, preds["d_end + d_min"], preds["d_end"], roc_auc_score, n=300, groups=g)
print(f"gain from adding d_min: {d:+.4f} [{ci[0]:+.4f}, {ci[1]:+.4f}]")

d_end                      CV ROC-AUC = 0.8842


d_end + d_min              CV ROC-AUC = 0.8879


d_end + d_min + d_start    CV ROC-AUC = 0.8924


gain from adding d_min: +0.0037 [+0.0027, +0.0045]


## 3. Far P passes (nearest defender ≥ 4 m at the pass frame): what happened earlier?

In [5]:
far = PN[PN.d_end >= 4].copy()
far["was_close"] = far.d_min < PFF_R_PRESSURED                         # a defender came within 2.74 m during the possession
far["approaching"] = (far.closing_max > 3) & ~far.was_close          # not close earlier, but someone closing fast at the pass
far["type"] = np.select([far.was_close, far.approaching], ["defender close earlier", "fast approach at pass"], "neither")
print(f"far passes: {len(far):,} (P {far.P.sum():,})")
tab = pd.crosstab(far.type, far.label, normalize="columns").round(3)
display(tab)
display(far.groupby("label")[["d_start", "d_min", "d_end", "t_min_before", "dur"]].median().round(2))
# how long before the pass was the closest approach, for far P that were close earlier
fp = far[(far.label == "P") & far.was_close]
print(f"far P with a close defender earlier: closest approach {fp.t_min_before.median():.2f} s before the pass (median), "
      f"min distance {fp.d_min.median():.2f} m")

far passes: 23,343 (P 2,838)


label,N,P
type,,
defender close earlier,0.031,0.093
fast approach at pass,0.159,0.221
neither,0.810,0.686


,d_start,d_min,d_end,t_min_before,dur
label,,,,,
N,10.09,7.27,7.63,0.03,1.37
P,6.59,5.32,5.92,0.00,0.87


far P with a close defender earlier: closest approach 1.69 s before the pass (median), min distance 1.34 m


In [6]:
# P-rate by pass-frame distance, split by whether a defender was within 2.74 m earlier in the possession
PN["was_close"] = PN.d_min < PFF_R_PRESSURED
PN["dbin"] = pd.cut(PN.d_end, [0, 1, 2, 3, 4, 5, 6, 8, 12, 100])
rate = PN.pivot_table(index="dbin", columns="was_close", values="P", aggfunc="mean", observed=True).round(3)
cnt = PN.pivot_table(index="dbin", columns="was_close", values="P", aggfunc="size", observed=True)
rate.columns, cnt.columns = ["P rate: never within 2.74 m", "P rate: within 2.74 m earlier"], ["n never", "n earlier"]
display(pd.concat([rate, cnt], axis=1))

fig, ax = plt.subplots(figsize=(5.4, 3.4))
x = [iv.mid for iv in rate.index][:-1]
ax.plot(x, rate.iloc[:-1, 0], marker="o", color="#9ecae1", label="no defender within 2.74 m during possession")
ax.plot(x, rate.iloc[:-1, 1], marker="o", color="#08306b", label="a defender within 2.74 m earlier in possession")
ax.axvline(2.74, ls=":", c="grey", lw=1)
ax.set_xlabel("nearest defender distance at the pass (m)"); ax.set_ylabel("share labelled 'pressured'")
ax.set_ylim(0, 1); ax.legend(frameon=False, fontsize=7)
ax.set_title("Analysts label the possession, not only the pass frame", fontsize=9)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "fig_possession_window.png"), dpi=300)

,P rate: never within 2.74 m,P rate: within 2.74 m earlier,n never,n earlier
dbin,,,,
"(0, 1]",NaN,0.931,NaN,2247.0
"(1, 2]",NaN,0.898,NaN,7086.0
"(2, 3]",0.638,0.778,1196.0,5412.0
"(3, 4]",0.434,0.408,4309.0,649.0
"(4, 5]",0.204,0.316,3861.0,351.0
"(5, 6]",0.146,0.332,3392.0,184.0
"(6, 8]",0.107,0.299,5090.0,184.0
"(8, 12]",0.084,0.252,5962.0,107.0
"(12, 100]",0.060,0.132,4136.0,76.0


## 4. Does the earlier pressure explain the higher failure of far P passes?

In [7]:
f2 = far.assign(close_earlier=np.where(far.was_close, "close earlier", "not close earlier"))
fail_tab = f2.pivot_table(index="close_earlier", columns="label", values="fail", aggfunc=["mean", "size"]).round(3)
fail_tab

mean          size      
label                  N      P      N     P
close_earlier                               
close earlier      0.132  0.242    638   264
not close earlier  0.089  0.259  19867  2574

## Save

In [8]:
moments.to_csv(os.path.join(OUT, "possession_moments_auc.csv"), index=False)
tab.to_csv(os.path.join(OUT, "possession_far_types.csv"))
rate.to_csv(os.path.join(OUT, "possession_prate_by_distance.csv"))
print("saved")

saved
